# Лекция: Подбор вида зависимости с помощью регрессии

**Дисциплина:** Введение в анализ больших данных  
**Задание 8** (адаптация с языка R на Python)

Задача: по данным **температура ~ глубина** сравнить три модели:
1. **Линейная:** $y = a_0 + a_1 x$
2. **Полиномиальная 2-й степени:** $y = a_0 + a_1 x + a_2 x^2$
3. **Логарифмическая:** $y = a_0 + a_1 log(x)$

Для каждой модели: анализ остатков, MSE, AIC.  
Сравнение моделей: `anova` (вложенные модели) и AIC.

В Python: **statsmodels** (`ols`, `anova_lm`, `AIC`).


## 0. Импорт библиотек


In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm
from statsmodels.stats.diagnostic import acorr_breusch_godfrey, het_breuschpagan

plt.rcParams['figure.figsize'] = (10, 6)
sns.set_style("whitegrid")
np.random.seed(42)
print("Библиотеки загружены")


## 1. Данные: температура и глубина

В задании используется файл **Osvech.csv**.  
Если у вас есть этот файл — загрузите его так:

```python
df = pd.read_csv("Osvech.csv")  # или sep=";", decimal=","
# переименуйте столбцы в depth и temp при необходимости
```

Ниже — **демонстрационный профиль** (типичная стратификация водоёма). Замените на свои данные.


In [ ]:
depth = np.array([0.5, 1, 2, 3, 4, 5, 6, 7, 8, 10, 12, 15, 18, 20, 25, 30, 35, 40])
temp = 22 - 8 * (1 - np.exp(-depth / 6)) - 0.05 * depth + np.random.normal(0, 0.3, len(depth))

df = pd.DataFrame({"depth": depth, "temp": temp})
print(df.round(2))
print("\nn =", len(df))


### График зависимости температуры от глубины


In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df["depth"], df["temp"], s=60, edgecolors="k", zorder=3)
plt.xlabel("Глубина")
plt.ylabel("Температура")
plt.title("Зависимость температуры от глубины")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


---
## 2. Три регрессионные модели


In [ ]:
m_lin = smf.ols("temp ~ depth", data=df).fit()
m_poly = smf.ols("temp ~ depth + I(depth ** 2)", data=df).fit()
m_log = smf.ols("temp ~ np.log(depth)", data=df).fit()

print("=== Линейная ===")
print(m_lin.summary().tables[1])
print("\n=== Полиномиальная (2) ===")
print(m_poly.summary().tables[1])
print("\n=== Логарифмическая ===")
print(m_log.summary().tables[1])


### Наложение кривых на данные


In [ ]:
x_grid = np.linspace(df["depth"].min(), df["depth"].max(), 200)
grid = pd.DataFrame({"depth": x_grid})

plt.figure(figsize=(9, 6))
plt.scatter(df["depth"], df["temp"], s=60, edgecolors="k", label="Данные", zorder=3)
plt.plot(x_grid, m_lin.predict(grid), label="Линейная", lw=2)
plt.plot(x_grid, m_poly.predict(grid), label="Полином 2", lw=2)
plt.plot(x_grid, m_log.predict(grid), label="Логарифмическая", lw=2)
plt.xlabel("Глубина")
plt.ylabel("Температура")
plt.title("Сравнение моделей")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


---
## 3. Анализ остатков, MSE и AIC

В R: `mean(lm$residuals^2)`, `AIC(lm)`


In [ ]:
def diagnose(model, name):
    resid = model.resid
    mse = np.mean(resid ** 2)
    aic = model.aic
    W, p_sw = stats.shapiro(resid)
    print(f"=== {name} ===")
    print(f"  R2      = {model.rsquared:.4f}")
    print(f"  Adj.R2  = {model.rsquared_adj:.4f}")
    print(f"  MSE     = {mse:.4f}")
    print(f"  AIC     = {aic:.2f}")
    print(f"  Shapiro p = {p_sw:.4f}  ({'нормальны' if p_sw > 0.05 else 'НЕ нормальны'})")
    print()
    return {"model": name, "R2": model.rsquared, "AdjR2": model.rsquared_adj,
            "MSE": mse, "AIC": aic, "Shapiro_p": p_sw}

rows = [
    diagnose(m_lin, "Линейная"),
    diagnose(m_poly, "Полином 2"),
    diagnose(m_log, "Логарифмическая"),
]
cmp = pd.DataFrame(rows)
print(cmp.round(4).to_string(index=False))


### Графики остатков


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, model, title in zip(axes, [m_lin, m_poly, m_log],
                            ["Линейная", "Полином 2", "Логарифмическая"]):
    ax.scatter(model.fittedvalues, model.resid, edgecolors="k", alpha=0.7)
    ax.axhline(0, color="red", ls="--")
    ax.set_xlabel("Fitted")
    ax.set_ylabel("Residuals")
    ax.set_title(title)
plt.suptitle("Остатки vs fitted", y=1.02)
plt.tight_layout()
plt.show()


---
## 4. Попарное сравнение моделей (ANOVA)

В R: `anova(object1, object2)`

Для **вложенных** моделей (линейная ⊂ полиномиальная) ANOVA проверяет, значимо ли усложнение модели.


In [ ]:
print("ANOVA: линейная vs полином 2")
print(anova_lm(m_lin, m_poly))
print()
print("Сравнение по AIC (меньше лучше):")
print(cmp[["model", "AIC", "MSE", "AdjR2"]].sort_values("AIC").to_string(index=False))


---
## 5. Какая модель лучше и почему?

Критерии выбора:
1. **AIC** — чем меньше, тем лучше
2. **MSE** — средняя квадратическая ошибка
3. **Adj. R²** — скорректированный коэффициент детерминации
4. **ANOVA** — для вложенных моделей
5. **Остатки** — без структуры, близки к нормальным

Выбирайте модель с наименьшим AIC и приемлемыми остатками.


In [ ]:
best = cmp.loc[cmp["AIC"].idxmin(), "model"]
print(f"По AIC лучшая модель: {best}")
print(cmp.sort_values("AIC")[["model", "AIC", "MSE", "AdjR2"]].round(4).to_string(index=False))


---
## Шпаргалка: R → Python

| Задача в R | Python |
|------------|--------|
| `lm(y ~ x)` | `smf.ols("y ~ x", data=df).fit()` |
| `lm(y ~ x + I(x^2))` | `smf.ols("y ~ x + I(x**2)", data=df).fit()` |
| `lm(y ~ log(x))` | `smf.ols("y ~ np.log(x)", data=df).fit()` |
| `mean(resid^2)` | `np.mean(model.resid**2)` |
| `AIC(lm)` | `model.aic` |
| `anova(m1, m2)` | `anova_lm(m1, m2)` |
| `summary(lm)` | `model.summary()` |

---
## Рекомендации

1. Для логарифмической модели **x > 0**.
2. AIC удобен для сравнения вложенных и невложенных моделей.
3. Подставьте свой `Osvech.csv`:

```python
df = pd.read_csv("Osvech.csv")
df = df.rename(columns={"глубина": "depth", "температура": "temp"})
```

**Удачи с выполнением Задания 8!**
